# PS: 3F
Using the Sign Language MNIST dataset, select any one of the 24 alphabet classes
(label Y-24) as an anomaly. Train an Autoencoder–Decoder model using the remaining
23 classes and detect the selected class as an anomaly based on reconstruction error.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import confusion_matrix, classification_report

from tensorflow.keras import models, layers

In [2]:
# LOADING THE DATASET
train = pd.read_csv("sign_mnist_train.csv")
test = pd.read_csv("sign_mnist_test.csv")

train.head()
train.shape

(27455, 785)

In [3]:
# PREPROCESS THE DATA
X_train = train.drop("label", axis = 1)
y_train = train["label"]

X_test = test.drop("label", axis=1 )
y_test = test["label"]

In [4]:
# NORMALIZE PIXEL VALUES
X_train = X_train / 255.0
X_test = X_test / 255.0

In [5]:
# SELECTING LABEL 24 AS ANOMALY
X_train = X_train[y_train != 24]

y_test = (y_test == 24).astype(int)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

Training shape: (26337, 784)
Testing shape: (7172, 784)


In [6]:
# DEFINING MODEL ARCHITECTURE
# ENCODER
encoder = models.Sequential([
    layers.Input(shape=(X_train.shape[1],)), 
    layers.Dense(128, activation = "relu"),
    layers.Dense(64, activation = "relu"),
    layers.Dense(32, activation = "relu"),
    layers.Dense(8, activation = "relu")
])

# DECODRE
decoder = models.Sequential([
    layers.Input(shape=(8,)),
    layers.Dense(32, activation = "relu"),
    layers.Dense(64, activation = "relu"),
    layers.Dense(128, activation = "relu"),
    layers.Dense(X_train.shape[1], activation = "sigmoid")
])

# AUTOENCODER
model = models.Sequential([
    encoder,
    decoder
])

#encoder.summary()
#decoder.summary()
#model.summary()

    

In [7]:
# COMPILE AND TRAIN THE MODLE
model.compile(
    optimizer = 'adam',
    loss = 'mse',
    metrics = ['mae']
)

history = model.fit(
    X_train, 
    X_train,
    epochs= 11,
    batch_size = 64,
    validation_split = 0.2
)

Epoch 1/11
330/330 ━━━━━━━━━━━━━━━━━━━━ 7s 13ms/step - loss: 0.0247 - mae: 0.1187 - val_loss: 0.0178 - val_mae: 0.0979
Epoch 2/11
330/330 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - loss: 0.0166 - mae: 0.0932 - val_loss: 0.0164 - val_mae: 0.0927
Epoch 3/11
330/330 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - loss: 0.0148 - mae: 0.0870 - val_loss: 0.0141 - val_mae: 0.0853
Epoch 4/11
330/330 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - loss: 0.0133 - mae: 0.0818 - val_loss: 0.0131 - val_mae: 0.0811
Epoch 5/11
330/330 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - loss: 0.0123 - mae: 0.0784 - val_loss: 0.0121 - val_mae: 0.0792
Epoch 6/11
330/330 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - loss: 0.0115 - mae: 0.0757 - val_loss: 0.0114 - val_mae: 0.0760
Epoch 7/11
330/330 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.0110 - mae: 0.0738 - val_loss: 0.0109 - val_mae: 0.0734
Epoch 8/11
330/330 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - loss: 0.0106 - mae: 0.0727 - val_loss: 0.0107 - val_mae: 0.0729
Epoch 9/11
330/330 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step 

In [8]:
# CALCULATE RECONSTRUCTION ERROR 
train_predictions = model.predict(X_test)

train_errors = np.mean(
    np.square(X_test - train_predictions), axis = 1
)

225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step   


In [9]:
# SETTING THE THRESHOLD
threshold = np.percentile(train_errors, 95)

print("Threshold: ", threshold)

Threshold:  0.02054031327961529


In [10]:
# DETECT ANOMALIES
test_predictions = model.predict(X_test)

test_errors = np.mean(
    np.square(X_test - test_predictions), axis = 1 
)

y_pred = (test_errors > threshold ).astype(int)

print(test_errors[:20])
print(y_pred[:20])

225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step 
0     0.007363
1     0.007480
2     0.009711
3     0.021645
4     0.012635
5     0.004703
6     0.005761
7     0.008423
8     0.010267
9     0.009622
10    0.019657
11    0.015840
12    0.007681
13    0.013697
14    0.007672
15    0.005327
16    0.003621
17    0.004715
18    0.019487
19    0.011294
dtype: float64
0     0
1     0
2     0
3     1
4     0
5     0
6     0
7     0
8     0
9     0
10    0
11    0
12    0
13    0
14    0
15    0
16    0
17    0
18    0
19    0
dtype: int64


In [11]:
# EVALUATE THE MODEL
print("Confusion Matrix")
print(confusion_matrix(y_test, y_pred))

print("Classification report")
print(classification_report(y_test, y_pred))

Confusion Matrix
[[6513  327]
 [ 300   32]]
Classification report
              precision    recall  f1-score   support

           0       0.96      0.95      0.95      6840
           1       0.09      0.10      0.09       332

    accuracy                           0.91      7172
   macro avg       0.52      0.52      0.52      7172
weighted avg       0.92      0.91      0.91      7172

